[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/91-analogical-reasoning/analogical_reasoning_workbook.ipynb)

# 91 · Analogical Reasoning — Solve by Generating Your Own Examples

## Webb et al. 2023 — Let the Model Recall Its Own Analogies

**Estimated time:** ~90 min

Analogical reasoning (Webb et al. 2023) turns the model into its own few-shot example generator. Instead of you providing examples, the model first recalls relevant analogies from its training, then uses those self-generated examples to solve the problem. This workbook builds the `generate_analogies → solve` LangGraph pipeline.

## Workshop Roadmap

| Part | Section | What you build |
|------|---------|----------------|
| 1 | Analogical vs. few-shot prompting | Understand the structural difference |
| 2 | The `generate_analogies` node | Prompt the model to recall its own examples |
| 3 | The `solve` node | Use recalled analogies to guide reasoning |
| 4 | Zero-shot vs. analogy-augmented | Side-by-side comparison on two problems |
| 5 | Full graph run | `generate_analogies → solve` LangGraph pipeline |
| 6 | Exercises | Extend and explore the technique |

**Reference:** Webb, T., Holyoak, K. J., & Lu, H. (2023). Emergent analogical reasoning in large language models. *Nature Human Behaviour*. https://www.nature.com/articles/s41562-023-01659-w

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langgraph", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
key_ok = bool(key) and key.startswith("sk-")
print(f"OPENAI_API_KEY present and valid-looking: {key_ok}")
if not key_ok:
    print("  ACTION REQUIRED — add your key before running LLM cells.")

## Part 1 — Analogical vs. few-shot prompting

**Key difference: who provides the examples?**

- **Few-shot prompting:** YOU provide 3–5 examples at inference time. This requires manual curation, domain knowledge, and the examples may not match the structural pattern of your target problem.
- **Analogical reasoning:** The MODEL generates its own examples first — like asking *"can you think of a similar problem you already know how to solve?"* The model draws from its training to recall structurally similar worked examples, then uses those as implicit few-shot context.

| Approach | Who picks examples | Effort | Relevance | Extra cost |
|----------|--------------------|--------|-----------|------------|
| Standard few-shot | You (human) | High — manual curation | Variable | None |
| Zero-shot CoT | Nobody | None | None | None |
| Analogical (Webb 2023) | The model itself | None — automatic | High — model picks structurally similar | +1 LLM call |

The core insight from Webb et al. is that large models have implicitly memorised thousands of worked examples during pre-training. Analogical prompting is simply a structured way to *retrieve and activate* that latent knowledge before tackling the target problem.

In [ ]:
# Show the structural difference between few-shot and analogical prompting
# Analogical: two-stage (analogies first, then solve)

problem = "A factory produces 240 widgets per 8-hour shift. How long to produce 900 widgets?"

# Standard few-shot: you hard-code examples
few_shot_prompt = '''Examples:
Q: A car goes 60 mph for 2 hours. How far? A: 120 miles.
Q: A printer prints 30 pages per minute. 150 pages takes how long? A: 5 minutes.
Q: {problem}
A:'''.format(problem=problem)

# Analogical stage 1: ask the model to recall similar problems
analogy_recall_prompt = '''Recall 2-3 problems that are structurally similar to the one below.
For each, write the problem AND solution. These should come from your training knowledge.

Target problem: {problem}

Similar problems from your knowledge:'''.format(problem=problem)

# Analogical stage 2: solve using recalled examples
analogy_solve_prompt = '''Here are similar problems and solutions you recalled:
{{recalled_analogies}}

Now solve this new problem using those analogies as a guide:
{problem}'''.format(problem=problem)

print("=== Few-shot prompt structure ===")
print(few_shot_prompt)
print()
print("=== Analogical stage 1: recall prompt structure ===")
print(analogy_recall_prompt)
print()
print("=== Analogical stage 2: solve prompt structure ===")
print(analogy_solve_prompt[:300])

## Part 2 — The `generate_analogies` node

**Prompting the model to recall its own examples**

The analogy generation prompt has three requirements:

1. **Ask for COMPLETE worked examples** — both the problem statement AND the step-by-step solution. A partial example (problem only) gives the solver no additional signal.
2. **Match the problem domain** — if your target is a rate problem, the recalled analogies should also be rate problems. The prompt should describe the target well enough that the model identifies the right structural pattern.
3. **Ask for 2–3 analogies** — not too few (one analogy may be a poor match), not too many (diminishing returns beyond 3; the prompt grows long and the solver may lose focus on the target).

The model is not being asked to *solve* anything in this step — only to *remember*. This is a retrieval-style call, not a reasoning call.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)  # slight diversity for richer analogies

ANALOGY_PROMPT = '''Generate 2-3 problems that are structurally similar to the target problem below.
For each, include: (1) the problem statement, (2) step-by-step solution, (3) final answer.
These should come from your own knowledge — do not reference the target problem's numbers.

Target problem: {problem}

Similar problems and solutions:'''

def generate_analogies(problem: str) -> str:
    """Ask the LLM to recall structurally similar problems from its training."""
    resp = llm.invoke([HumanMessage(content=ANALOGY_PROMPT.format(problem=problem))])
    return resp.content.strip()

target = "A factory produces 240 widgets per 8-hour shift. How long to produce 900 widgets?"
print(f"Target problem: {target}\n")
print("Recalled analogies:\n")
analogies = generate_analogies(target)
print(analogies)

## Part 3 — The `solve` node

**How recalled analogies shape the reasoning**

The `solve` node receives both the target problem AND the recalled analogies. The analogies serve as implicit few-shot examples that were chosen by the model itself — they tend to be more structurally relevant than human-chosen examples because the model picked them to be similar.

Three things happen when the solver sees recalled analogies:

1. **Pattern activation** — the solver sees the same structural pattern (e.g. rate × time = quantity) applied 2–3 times, reinforcing the correct approach.
2. **Format guidance** — the worked examples show what a complete step-by-step solution looks like, nudging the solver to produce the same level of detail.
3. **Implicit calibration** — if the solver's first instinct was an incorrect approach, seeing a worked analogy can redirect it before it starts writing.

The solve prompt must explicitly connect the analogies to the target problem — telling the model to "use the patterns you identified above" ensures the analogies are not ignored.

In [ ]:
SOLVE_WITH_ANALOGIES_PROMPT = '''You recalled these similar problems and solutions:

{analogies}

Now solve the target problem using the patterns you identified above.
Show your reasoning step by step. End with "Answer: <final answer>".

Target problem: {problem}'''

def solve_with_analogies(problem: str, analogies: str) -> str:
    """Solve the target problem using recalled analogies as implicit examples."""
    prompt = SOLVE_WITH_ANALOGIES_PROMPT.format(analogies=analogies, problem=problem)
    resp = llm.invoke([HumanMessage(content=prompt)])
    return resp.content.strip()

print(f"Problem: {target}\n")
solution = solve_with_analogies(target, analogies)
print("Solution with analogical reasoning:\n")
print(solution)

## Part 4 — Compare: zero-shot vs. analogy-augmented on the same problem

**Seeing the difference directly**

The clearest way to see analogical reasoning's value is to compare zero-shot chain-of-thought (CoT) against analogy-augmented on the same problem. The gap is most visible on:

- **Abstract pattern problems** — where the model may not immediately recognise the governing rule (e.g. geometric sequences, work-rate problems).
- **Problems with misleading surface features** — where the wording suggests a wrong approach, but a structurally similar analogy clarifies the correct one.
- **Multi-step problems** — where analogies provide a sequencing scaffold the solver would otherwise have to construct from scratch.

Run both approaches below and compare the final answers. Note whether the *reasoning path* changes, not just the answer.

In [ ]:
ZERO_SHOT_PROMPT = '''Solve this problem step by step. End with "Answer: <final answer>".

Problem: {problem}'''

def zero_shot_solve(problem: str) -> str:
    resp = llm.invoke([HumanMessage(content=ZERO_SHOT_PROMPT.format(problem=problem))])
    return resp.content.strip()

test_problems = [
    "A factory produces 240 widgets per 8-hour shift. How long to produce 900 widgets?",
    "A series: 2, 6, 18, 54, ... What is the 7th term?"
]

for prob in test_problems:
    print(f"Problem: {prob}")
    print("-" * 60)

    zs = zero_shot_solve(prob)
    an = generate_analogies(prob)
    ar = solve_with_analogies(prob, an)

    print("Zero-shot CoT answer:")
    # Show last 2 lines (usually contains the final answer)
    zs_lines = [l for l in zs.split('\n') if l.strip()]
    for line in zs_lines[-2:]:
        print(f"  {line}")

    print("Analogy-augmented answer:")
    ar_lines = [l for l in ar.split('\n') if l.strip()]
    for line in ar_lines[-2:]:
        print(f"  {line}")
    print()

## Part 5 — Full graph run: `generate_analogies → solve`

**The complete LangGraph pipeline**

Now we wire the two functions into a proper LangGraph state machine:

```
START → generate_analogies_node → solve_node → END
```

The `AnalogicalState` carries three fields through the graph:
- `problem` — the input, set at invocation time
- `analogies` — populated by `generate_analogies_node`
- `solution` — populated by `solve_node`

Each node only reads the state fields it needs and returns a dict with only the fields it updates — the graph merges them automatically.

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, END

class AnalogicalState(TypedDict):
    problem: str
    analogies: str    # recalled by generate_analogies node
    solution: str     # produced by solve node

def generate_analogies_node(state: AnalogicalState) -> dict:
    """Node 1: ask the model to recall structurally similar problems."""
    print("Generating analogies...")
    recalled = generate_analogies(state["problem"])
    # Print a preview of recalled analogies
    preview = recalled[:200].replace('\n', ' ')
    print(f"  Recalled {len(recalled.split(chr(10)))} lines of analogies: {preview}...")
    return {"analogies": recalled}

def solve_node(state: AnalogicalState) -> dict:
    """Node 2: solve using the recalled analogies as context."""
    print("Solving with analogies...")
    solution = solve_with_analogies(state["problem"], state["analogies"])
    # Extract final answer line
    lines = [l for l in solution.split('\n') if l.strip()]
    answer_line = next((l for l in reversed(lines) if 'answer' in l.lower()), lines[-1] if lines else "")
    print(f"  {answer_line}")
    return {"solution": solution}

g = StateGraph(AnalogicalState)
g.add_node("generate_analogies_node", generate_analogies_node)
g.add_node("solve_node", solve_node)
g.set_entry_point("generate_analogies_node")
g.add_edge("generate_analogies_node", "solve_node")
g.add_edge("solve_node", END)
app = g.compile()

problems = [
    "If 6 workers build a wall in 12 days, how many days for 9 workers to build the same wall?",
    "A snail climbs 3 feet per day but slides back 1 foot each night. How many days to climb out of a 10-foot well?"
]

for problem in problems:
    print(f"\nProblem: {problem}")
    print("=" * 60)
    result = app.invoke({"problem": problem, "analogies": "", "solution": ""})
    print("\nFull solution:")
    print(result["solution"][:600])

## Part 6 — Exercises

### Exercise (a) — Domain-constrained analogy prompts

Modify `ANALOGY_PROMPT` to constrain the domain of recalled analogies. For example:
- Add `"from physics"` → analogies about forces, velocity, energy
- Add `"from software engineering"` → analogies about throughput, latency, concurrency
- Add `"from biology"` → analogies about cell division rates, population growth

Observe how the domain constraint shifts the recalled examples while the structural pattern (rate × time = quantity) stays constant.

### Exercise (b) — Inspect what analogies the model recalls across problem types

Run `generate_analogies` on three structurally different problem types and compare:
1. A rate problem (time / work / speed)
2. A geometric sequence problem
3. A logic puzzle

Notice that analogies cluster around the **structural pattern** (rate × time = work, geometric series, etc.) not the **surface content** (factories, snails, number sequences). This is the key insight from Webb et al. — the model abstracts structure, not surface.

In [ ]:
# Exercise (a): Domain-constrained analogy prompt
# TODO: modify ANALOGY_PROMPT_DOMAIN below to add a domain constraint
# and observe how analogies shift while the structural pattern stays the same

ANALOGY_PROMPT_DOMAIN = '''Generate 2-3 problems that are structurally similar to the target problem below.
Draw your examples specifically from {domain}.
For each, include: (1) the problem statement, (2) step-by-step solution, (3) final answer.

Target problem: {problem}

Similar problems and solutions from {domain}:'''

def generate_analogies_domain(problem: str, domain: str) -> str:
    """Generate analogies constrained to a specific domain."""
    prompt = ANALOGY_PROMPT_DOMAIN.format(problem=problem, domain=domain)
    resp = llm.invoke([HumanMessage(content=prompt)])
    return resp.content.strip()

# TODO: run this cell and compare analogies across domains
# example_problem = "A factory produces 240 widgets per 8-hour shift. How long to produce 900 widgets?"
# for domain in ["physics", "software engineering", "biology"]:
#     print(f"\n--- Domain: {domain} ---")
#     print(generate_analogies_domain(example_problem, domain)[:400])

print("Exercise (a) starter ready.")
print("Uncomment the TODO block above and run to compare domain-constrained analogies.")


# Exercise (b): Inspect analogies across three structurally different problem types
# TODO: run generate_analogies on each problem type and compare what the model recalls

exercise_b_problems = [
    ("rate",     "A tap fills a tank in 4 hours. A drain empties it in 6 hours. How long to fill the tank with both open?"),
    ("sequence", "A series: 3, 9, 27, 81, ... What is the 6th term?"),
    ("logic",    "Alice is older than Bob. Bob is older than Carol. Carol is older than Dave. Who is oldest?"),
]

# TODO: uncomment to run and compare
# for label, prob in exercise_b_problems:
#     print(f"\n=== Problem type: {label} ===")
#     print(f"Problem: {prob}")
#     print("Recalled analogies:")
#     print(generate_analogies(prob)[:500])

print("\nExercise (b) starter ready.")
print("Uncomment the TODO block above and run to inspect cross-domain structural clustering.")

## Answer Key

### Exercise (a) — Domain-constrained analogies

Domain-constrained analogy prompts improve precision when the problem is domain-specific. Adding `"from software engineering principles"` to the prompt causes the recalled analogies to shift from generic math (widget factories, car speeds) to system design patterns (request throughput, queue drain rate, pipeline latency). Yet the underlying structure — `rate × time = total` — is preserved across all domains.

This means analogical prompting can be steered to match the register of the target problem: a SWE interviewer asking about system throughput will get more relevant analogies from the software domain than from a generic rate-math domain, even though the mathematics is identical.

**Key takeaway:** Domain constraints change the *vocabulary* of analogies without changing the *structure*. Use domain constraints when the solver needs examples in the same conceptual language as the problem.

### Exercise (b) — Analogies cluster on structure, not surface

When you run `generate_analogies` across the three problem types, you will observe:

- **Rate problems** → analogies involve combined work rates, speed/distance, fluid flow — all governed by `rate × time = amount`.
- **Geometric sequences** → analogies involve compound interest, population doubling, radioactive decay — all governed by `a × r^(n-1)`.
- **Logic puzzles** → analogies involve transitive ordering (taller/shorter, heavier/lighter, faster/slower) — all governed by transitivity of a total order.

The surface content (factories, snails, people named Alice) is completely different, but the structural pattern is perfectly preserved. This is exactly the finding in Webb et al. 2023: large language models perform analogical reasoning by extracting and transferring abstract relational structure, not by matching surface-level features. This is why analogical prompting transfers across domains — the model's recalled analogies are structurally matched even when the topic is completely different.